# Model 4: Outfit Recommendation
**Dataset:** Fashion Product Images (Kaggle) — metadata CSV (~44,000 products)  
**Architecture:** Content-Based Filtering (TF-IDF + Cosine Similarity) + rule-based scoring  
**Output:** `outfit_recommender.pkl`  
**Target:** Trả về ≥ 5 outfit phù hợp với input

---
**Input:** skin_tone + body_shape + occasion + gender  
**Output:** Danh sách sản phẩm phù hợp (top-K)

**Dataset:** `https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-dataset`
(Dùng file `styles.csv` — không cần download ảnh)

In [ ]:
# ============================================================
# CELL 1: Install
# ============================================================
!pip install scikit-learn pandas numpy matplotlib faiss-cpu --quiet

import os, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")

In [ ]:
# ============================================================
# CELL 2: Load dataset
# ============================================================
import glob

# Try hardcoded paths first, then search dynamically
KAGGLE_PATHS = [
    "/kaggle/input/fashion-product-images-small/styles.csv",
    "/kaggle/input/fashion-product-images-dataset/fashion-dataset/styles.csv",
    "./data/fashion-product/styles.csv",
]

CSV_PATH = None
for p in KAGGLE_PATHS:
    if os.path.exists(p):
        CSV_PATH = p
        break

# Fallback: search anywhere under /kaggle/input/
if CSV_PATH is None:
    matches = glob.glob("/kaggle/input/**/styles.csv", recursive=True)
    if matches:
        CSV_PATH = matches[0]
        print(f"Found via search: {CSV_PATH}")

if CSV_PATH is None:
    # Show what IS available to help debug
    print("styles.csv not found. Available /kaggle/input/ contents:")
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            print("  ", os.path.join(root, f))
    raise FileNotFoundError(
        "styles.csv not found. See available files above.\n"
        "On Kaggle: Click '+Add Input' → search 'fashion-product-images-small' → Add"
    )

print(f"Using: {CSV_PATH}")
df = pd.read_csv(CSV_PATH, on_bad_lines='skip')
print(f"Total products: {len(df)}")
print(df.head())
print("\nColumns:", df.columns.tolist())

In [ ]:
# ============================================================
# CELL 3: Explore & clean
# ============================================================
print("Gender:", df['gender'].unique())
print("masterCategory:", df['masterCategory'].unique())
print("subCategory:", df['subCategory'].value_counts().head(10))
print("articleType:", df['articleType'].value_counts().head(15))
print("baseColour:", df['baseColour'].value_counts().head(15))
print("usage:", df['usage'].value_counts())
print("season:", df['season'].value_counts())

# Keep only Apparel & Footwear
df = df[df['masterCategory'].isin(['Apparel', 'Footwear', 'Accessories'])].copy()
df = df.dropna(subset=['articleType', 'baseColour', 'gender'])
print(f"\nCleaned: {len(df)} products")

In [ ]:
# ============================================================
# CELL 4: Build content tags for each product
# ============================================================
def build_tags(row):
    """Combine metadata into a searchable text string."""
    parts = [
        str(row.get('gender', '')).lower(),
        str(row.get('masterCategory', '')).lower(),
        str(row.get('subCategory', '')).lower(),
        str(row.get('articleType', '')).lower(),
        str(row.get('baseColour', '')).lower(),
        str(row.get('season', '')).lower(),
        str(row.get('usage', '')).lower(),
        str(row.get('productDisplayName', '')).lower()
    ]
    return ' '.join([p for p in parts if p != 'nan'])

df['tags'] = df.apply(build_tags, axis=1)
print("Sample tags:")
print(df['tags'].head(3).values)

In [ ]:
# ============================================================
# CELL 5: TF-IDF vectorization
# ============================================================
print("Building TF-IDF matrix...")
tfidf = TfidfVectorizer(max_features=3000, stop_words='english', ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df['tags'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
# tfidf_matrix: (num_products, num_features)

In [ ]:
# ============================================================
# CELL 6: Rule-based scoring layer
# ============================================================
# Map body shape → preferred article types
BODY_SHAPE_PREFERENCES = {
    'Hourglass':          ['Wrap Dress', 'Bodycon', 'Fitted Blazer', 'High-Waist Jeans', 
                           'Pencil Skirt', 'Belt'],
    'Pear':               ['A-Line Skirt', 'Wide Leg Pants', 'Off Shoulder Tops', 
                           'Flared Jeans', 'Maxi Dress'],
    'Apple':              ['Empire Waist', 'V-Neck', 'Flowy Tops', 'Wide Leg Trousers', 
                           'Maxi Skirt'],
    'Rectangle':          ['Peplum Tops', 'Ruffle', 'Belted Dress', 'Crop Top', 
                           'Layered Look', 'Midi Skirt'],
    'InvertedTriangle':   ['A-Line Skirt', 'Flared Pants', 'V-Neck', 'Ruffled Skirt', 
                           'Wide Leg Jeans']
}

# Map Fitzpatrick tone → preferred color families
SKIN_TONE_COLORS = {
    1: ['pink', 'lavender', 'soft blue', 'mint', 'rose', 'white', 'light'],
    2: ['coral', 'peach', 'warm yellow', 'light green', 'warm pink', 'cream'],
    3: ['coral', 'peach', 'green', 'warm white', 'nude', 'camel'],
    4: ['olive', 'orange', 'mustard', 'terracotta', 'rust', 'gold', 'brown'],
    5: ['orange', 'rust', 'gold', 'dark green', 'burgundy', 'rich purple'],
    6: ['black', 'white', 'navy', 'royal blue', 'bright red', 'emerald', 'yellow']
}

OCCASION_KEYWORDS = {
    'casual':   ['casual', 'sport', 'everyday', 'denim', 'tshirt', 't-shirt'],
    'formal':   ['formal', 'office', 'blazer', 'shirt', 'trouser', 'suit'],
    'party':    ['party', 'evening', 'cocktail', 'dress', 'heels', 'sequin'],
    'sport':    ['sport', 'gym', 'athletic', 'activewear', 'running', 'yoga'],
    'outdoor':  ['outdoor', 'casual', 'jacket', 'hoodie', 'denim']
}

print("Rule maps defined.")

In [ ]:
# ============================================================
# CELL 7: Recommendation function
# ============================================================
def build_query(skin_tone: int, body_shape: str, occasion: str, gender: str = 'Women') -> str:
    """Build a query string from user profile."""
    parts = [gender.lower(), occasion.lower()]

    # Add skin-tone-preferred colors
    colors = SKIN_TONE_COLORS.get(skin_tone, [])
    parts.extend(colors[:3])

    # Add body-shape-preferred styles
    styles = BODY_SHAPE_PREFERENCES.get(body_shape, [])
    parts.extend([s.lower() for s in styles[:3]])

    # Add occasion keywords
    occ_kws = OCCASION_KEYWORDS.get(occasion, [])
    parts.extend(occ_kws[:3])

    return ' '.join(parts)


def recommend_outfits(
    skin_tone: int,
    body_shape: str,
    occasion: str,
    gender: str = 'Women',
    top_k: int = 10
):
    """
    Returns top-K product recommendations.
    """
    # 1. Build query
    query = build_query(skin_tone, body_shape, occasion, gender)

    # 2. TF-IDF similarity
    query_vec  = tfidf.transform([query])
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # 3. Filter by gender
    gender_mask = df['gender'].str.lower().isin(
        [gender.lower(), 'unisex', 'boys', 'girls']
    )
    sim_scores[~gender_mask.values] = 0

    # 4. Top-K
    top_indices = sim_scores.argsort()[::-1][:top_k * 3]  # get more, then filter
    results = []
    seen_types = set()

    for idx in top_indices:
        row = df.iloc[idx]
        article = str(row.get('articleType', ''))
        # Diversify: max 2 per article type
        key = article
        if seen_types.count(key) if hasattr(seen_types, 'count') else \
           sum(1 for r in results if r['articleType'] == article) >= 2:
            continue
        results.append({
            'product_id': int(row['id']) if 'id' in row else idx,
            'name': str(row.get('productDisplayName', 'N/A')),
            'articleType': article,
            'color': str(row.get('baseColour', 'N/A')),
            'gender': str(row.get('gender', 'N/A')),
            'usage': str(row.get('usage', 'N/A')),
            'season': str(row.get('season', 'N/A')),
            'similarity_score': round(float(sim_scores[idx]), 4)
        })
        if len(results) >= top_k:
            break

    return {
        'query_used': query,
        'total_found': len(results),
        'recommendations': results
    }


# Test
result = recommend_outfits(
    skin_tone=3,
    body_shape='Pear',
    occasion='casual',
    gender='Women',
    top_k=8
)
print(f"Query: {result['query_used']}")
print(f"Found: {result['total_found']} recommendations\n")
for i, r in enumerate(result['recommendations'], 1):
    print(f"  {i:02d}. [{r['articleType']}] {r['name']} | {r['color']} | score={r['similarity_score']}")

In [ ]:
# ============================================================
# CELL 8: Evaluate - Precision@K test
# ============================================================
# Giả lập test: verify ít nhất 5 results phù hợp (tiêu chí requirement)

test_cases = [
    {'skin_tone': 1, 'body_shape': 'Hourglass',        'occasion': 'formal',  'gender': 'Women'},
    {'skin_tone': 4, 'body_shape': 'Pear',             'occasion': 'casual',  'gender': 'Women'},
    {'skin_tone': 6, 'body_shape': 'Rectangle',        'occasion': 'party',   'gender': 'Women'},
    {'skin_tone': 2, 'body_shape': 'Apple',            'occasion': 'sport',   'gender': 'Men'},
    {'skin_tone': 5, 'body_shape': 'InvertedTriangle', 'occasion': 'casual',  'gender': 'Men'},
]

print("=" * 60)
print("Precision@5 Evaluation")
print("=" * 60)
all_pass = True

for case in test_cases:
    res = recommend_outfits(**case, top_k=10)
    n_found = len(res['recommendations'])
    passed = n_found >= 5
    if not passed:
        all_pass = False
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] skin={case['skin_tone']} body={case['body_shape']:18s} "
          f"occ={case['occasion']:8s} → {n_found} results")

print("\nAll tests passed!" if all_pass else "\nSome tests failed. Check data.")

In [ ]:
# ============================================================
# CELL 9: Save model
# ============================================================
# Lưu TF-IDF matrix dạng sparse để tiết kiệm bộ nhớ
import scipy.sparse as sp

sp.save_npz('tfidf_matrix.npz', tfidf_matrix)

with open('outfit_recommender.pkl', 'wb') as f:
    pickle.dump({
        'tfidf_vectorizer': tfidf,
        'product_data': df[['id', 'productDisplayName', 'articleType',
                             'baseColour', 'gender', 'usage', 'season', 'masterCategory']].to_dict('records'),
        'skin_tone_colors': SKIN_TONE_COLORS,
        'body_shape_preferences': BODY_SHAPE_PREFERENCES,
        'occasion_keywords': OCCASION_KEYWORDS,
    }, f)

print("Saved: outfit_recommender.pkl")
print("Saved: tfidf_matrix.npz")
print(f"Model covers {len(df):,} products.")

In [ ]:
# ============================================================
# CELL 10: Visual analysis - top colors per occasion
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, occ in zip(axes, ['Casual', 'Formal', 'Sports']):
    mask = df['usage'].str.lower() == occ.lower()
    top_colors = df.loc[mask, 'baseColour'].value_counts().head(10)
    ax.barh(top_colors.index[::-1], top_colors.values[::-1], color='steelblue')
    ax.set_title(f'Top Colors - {occ}')
    ax.set_xlabel('Count')

plt.tight_layout()
plt.savefig('top_colors_by_occasion.png', dpi=100)
plt.show()